# Automated Project: Planning, Estimation, and Allocation

In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Using python3.11.3

!pip3 install -r ../requirements.txt

/Users/namitjain/.zshenv:.:1: no such file or directory: /Users/namitjain/.cargo/env

[notice] A new release of pip is available: 25.0 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


In [3]:
# Load environment variables
from helper import load_env
load_env()

import os
import yaml
from crewai import Agent, Task, Crew

In [9]:
os.environ['OPENAI_MODEL_NAME'] = 'gpt-4o-mini'

In [5]:
# Define file paths for YAML configurations
files = {
    'agents': 'config/agents.yaml',
    'tasks': 'config/tasks.yaml'
}

# Load configurations from YAML files
configs = {}
for config_type, file_path in files.items():
    with open(file_path, 'r') as file:
        configs[config_type] = yaml.safe_load(file)

# Assign loaded configurations to specific variables
agents_config = configs['agents']
tasks_config = configs['tasks']

In [6]:
from typing import List
from pydantic import BaseModel, Field

class TaskEstimate(BaseModel):
    task_id: str = Field(..., description="Unique identifier for the task") 
    task_name: str = Field(..., description="Name of the task")
    estimated_time_hours: float = Field(..., description="Estimated time to complete the task in hours")
    required_resources: List[str] = Field(..., description="List of resources required to complete the task")

class Milestone(BaseModel):
    milestone_name: str = Field(..., description="Name of the milestone")
    tasks: List[str] = Field(..., description="List of task IDs associated with this milestone")

class ProjectPlan(BaseModel):
    tasks: List[TaskEstimate] = Field(..., description="List of tasks with their estimates")
    milestones: List[Milestone] = Field(..., description="List of project milestones")

In [7]:
# Creating Agents
project_planning_agent = Agent(
  config=agents_config['project_planning_agent']
)

estimation_agent = Agent(
  config=agents_config['estimation_agent']
)

resource_allocation_agent = Agent(
  config=agents_config['resource_allocation_agent']
)

# Creating Tasks
task_breakdown = Task(
  config=tasks_config['task_breakdown'],
  agent=project_planning_agent
)

time_resource_estimation = Task(
  config=tasks_config['time_resource_estimation'],
  agent=estimation_agent
)

resource_allocation = Task(
  config=tasks_config['resource_allocation'],
  agent=resource_allocation_agent,
  output_pydantic=ProjectPlan # This is the structured output we want
)

# Creating Crew
crew = Crew(
  agents=[
    project_planning_agent,
    estimation_agent,
    resource_allocation_agent
  ],
  tasks=[
    task_breakdown,
    time_resource_estimation,
    resource_allocation
  ],
  verbose=True
)

In [8]:
from IPython.display import display, Markdown

project = 'Website'
industry = 'Technology'
project_objectives = 'Create a website for a small business'
team_members = """
- John Doe (Project Manager)
- Jane Doe (Software Engineer)
- Bob Smith (Designer)
- Alice Johnson (QA Engineer)
- Tom Brown (QA Engineer)
"""
project_requirements = """
- Create a responsive design that works well on desktop and mobile devices
- Implement a modern, visually appealing user interface with a clean look
- Develop a user-friendly navigation system with intuitive menu structure
- Include an "About Us" page highlighting the company's history and values
- Design a "Services" page showcasing the business's offerings with descriptions
- Create a "Contact Us" page with a form and integrated map for communication
- Implement a blog section for sharing industry news and company updates
- Ensure fast loading times and optimize for search engines (SEO)
- Integrate social media links and sharing capabilities
- Include a testimonials section to showcase customer feedback and build trust
"""

# Format the dictionary as Markdown for a better display in Jupyter Lab
formatted_output = f"""
**Project Type:** {project}

**Project Objectives:** {project_objectives}

**Industry:** {industry}

**Team Members:**
{team_members}
**Project Requirements:**
{project_requirements}
"""
# Display the formatted output as Markdown
display(Markdown(formatted_output))


**Project Type:** Website

**Project Objectives:** Create a website for a small business

**Industry:** Technology

**Team Members:**

- John Doe (Project Manager)
- Jane Doe (Software Engineer)
- Bob Smith (Designer)
- Alice Johnson (QA Engineer)
- Tom Brown (QA Engineer)

**Project Requirements:**

- Create a responsive design that works well on desktop and mobile devices
- Implement a modern, visually appealing user interface with a clean look
- Develop a user-friendly navigation system with intuitive menu structure
- Include an "About Us" page highlighting the company's history and values
- Design a "Services" page showcasing the business's offerings with descriptions
- Create a "Contact Us" page with a form and integrated map for communication
- Implement a blog section for sharing industry news and company updates
- Ensure fast loading times and optimize for search engines (SEO)
- Integrate social media links and sharing capabilities
- Include a testimonials section to showcase customer feedback and build trust



In [10]:
# The given Python dictionary
inputs = {
  'project_type': project,
  'project_objectives': project_objectives,
  'industry': industry,
  'team_members': team_members,
  'project_requirements': project_requirements
}

# Run the crew
result = crew.kickoff(
  inputs=inputs
)

# Agent: The Ultimate Project Planner
## Task: Carefully analyze the project_requirements for the Website project and break them down into individual tasks. Define each task's scope in detail, set achievable timelines, and ensure that all dependencies are accounted for:

- Create a responsive design that works well on desktop and mobile devices
- Implement a modern, visually appealing user interface with a clean look
- Develop a user-friendly navigation system with intuitive menu structure
- Include an "About Us" page highlighting the company's history and values
- Design a "Services" page showcasing the business's offerings with descriptions
- Create a "Contact Us" page with a form and integrated map for communication
- Implement a blog section for sharing industry news and company updates
- Ensure fast loading times and optimize for search engines (SEO)
- Integrate social media links and sharing capabilities
- Include a testimonials section to showcase customer feedback and build tru

In [11]:
import pandas as pd

costs = 0.150 * (crew.usage_metrics.prompt_tokens + crew.usage_metrics.completion_tokens) / 1_000_000
print(f"Total costs: ${costs:.4f}")

# Convert UsageMetrics instance to a DataFrame
df_usage_metrics = pd.DataFrame([crew.usage_metrics.dict()])
df_usage_metrics

Total costs: $0.0013


,total_tokens,prompt_tokens,cached_prompt_tokens,completion_tokens,successful_requests
0,8455,4021,0,4434,3


In [12]:
result.pydantic.dict()

{'tasks': [{'task_id': '1.1',
   'task_name': 'Initial meeting to discuss requirements and expectations',
   'estimated_time_hours': 8.0,
   'required_resources': ['John Doe', 'Stakeholders']},
  {'task_id': '1.2',
   'task_name': 'Detailed documentation of the project requirements',
   'estimated_time_hours': 16.0,
   'required_resources': ['John Doe']},
  {'task_id': '2.1',
   'task_name': 'Create wireframes for all pages',
   'estimated_time_hours': 24.0,
   'required_resources': ['Bob Smith']},
  {'task_id': '2.2',
   'task_name': 'Develop responsive design mockups',
   'estimated_time_hours': 40.0,
   'required_resources': ['Bob Smith']},
  {'task_id': '2.3',
   'task_name': 'Review and approval of design mockups',
   'estimated_time_hours': 16.0,
   'required_resources': ['Bob Smith', 'John Doe', 'Team']},
  {'task_id': '3.1',
   'task_name': 'Setup development environment',
   'estimated_time_hours': 8.0,
   'required_resources': ['Jane Doe']},
  {'task_id': '3.2',
   'task_name

In [29]:
tasks = result.pydantic.dict()['tasks']
df_tasks = pd.DataFrame(tasks)

# Display the DataFrame as an HTML table
df_tasks.style.set_table_attributes('border="1"').set_caption("Task Details").set_table_styles(
    [{'selector': 'th, td', 'props': [('font-size', '120%')]}]
)

,task_id,task_name,estimated_time_hours,required_resources
0,1.1,Initial meeting to discuss requirements and expectations,8.000000,"['John Doe', 'Stakeholders']"
1,1.2,Detailed documentation of the project requirements,16.000000,['John Doe']
2,2.1,Create wireframes for all pages,24.000000,['Bob Smith']
3,2.2,Develop responsive design mockups,40.000000,['Bob Smith']
4,2.3,Review and approval of design mockups,16.000000,"['Bob Smith', 'John Doe', 'Team']"
5,3.1,Setup development environment,8.000000,['Jane Doe']
6,3.2,Create a responsive design framework,24.000000,"['Jane Doe', 'Bob Smith']"
7,3.3,Implement User Interface designs,40.000000,"['Jane Doe', 'Bob Smith']"
8,3.4,Develop navigation system,16.000000,['Jane Doe']
9,3.5,"Code various pages (About Us, Services, Contact Us including form and map)",72.000000,['Jane Doe']


In [14]:
milestones = result.pydantic.dict()['milestones']
df_milestones = pd.DataFrame(milestones)

# Display the DataFrame as an HTML table
df_milestones.style.set_table_attributes('border="1"').set_caption("Task Details").set_table_styles(
    [{'selector': 'th, td', 'props': [('font-size', '120%')]}]
)

,milestone_name,tasks
0,Planning and Requirements Gathering,"['1.1', '1.2']"
1,Design Phase,"['2.1', '2.2', '2.3']"
2,Development Phase,"['3.1', '3.2', '3.3', '3.4', '3.5', '3.6', '3.7', '3.8', '3.9']"
3,Testing Phase,"['4.1', '4.2', '4.3', '4.4', '4.5']"
4,Deployment and Review,"['5.1', '5.2', '5.3', '5.4']"
5,Post-Launch Analysis and Support,"['6.1', '6.2']"
